<a href="https://colab.research.google.com/github/anawag/pandas-numpy/blob/main/Estudo_Alunos_e_H%C3%A1bitos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importando as fontes de dados

In [ ]:
import pandas as pd
import io

alunos = pd.read_csv("alunos_realista.csv")
escolas = pd.read_csv("escolas_realista.csv")
habitos = pd.read_csv("habitos_estudo_realista.csv")
rendimento = pd.read_csv("rendimento_academico_realista.csv")

- Base de Dados - Alunos

In [ ]:
import pandas as pd


alunos = alunos.astype({
    "ID_Aluno": "int",
    "Nome": "str",
    "Genero": "str",
    "ID_Escola": "int"
})

alunos["Data_Nascimento"] = pd.to_datetime(
    alunos["Data_Nascimento"],
    dayfirst=True,
    errors="coerce"
)


data_min = pd.Timestamp("1920-10-01")
data_max = pd.Timestamp("2025-10-20")

limite_data = (alunos["Data_Nascimento"] < data_min) | (alunos["Data_Nascimento"] > data_max)

alunos_validos = alunos.loc[limite_data]
alunos_invalidos = alunos.loc[limite_data]

alunos["Data_Nascimento"] = alunos["Data_Nascimento"].dt.strftime("%d/%m/%Y")

if alunos["Nome"].duplicated().any() and alunos["ID_Aluno"].duplicated().any():
    duplicados = alunos[alunos["Nome"].duplicated(keep=False)], alunos[alunos["ID_Aluno"].duplicated(keep=False)]
    print(duplicados[["Nome"], ["ID_Aluno"]])
else:
    print("Nenhum nome duplicado")


print("===== RESUMO DAS DATAS DE NASCIMENTO =====")
print(f"Total de registros: {len(alunos)}")
print(f"Datas válidas: {len(alunos_validos)}")
print(f"Datas inválidas: {len(alunos_invalidos)}")

if not alunos_invalidos.empty:
    print("\nDatas de nascimento inválidas encontradas:")

- Base de Dados - Escolas

In [ ]:
escolas = escolas.astype({
    "ID_Escola": "int",
    "Nome_Escola": "str",
    "Tipo_Escola": "str",
    "Localizacao": "str"
})

if escolas["ID_Escola"].duplicated().any() and escolas["Nome_Escola"].duplicated().any():
    duplicados_escola = escolas[escolas["ID_Escola"].duplicated(keep=False)], escolas[escolas["Nome_Escola"].duplicated(keep=False)]
    print(f"Existem registros duplicados: {duplicados_escola["ID_Escola"],["Nome_Escola"]}")
else:
    print("Não existem registros duplicados.")

- Base de Dados - Hábitos

In [ ]:
habitos['Horas_Estudo_Diario'] = habitos['Horas_Estudo_Diario'].fillna(0)
habitos['Tempo_Midias_Sociais_Minutos'] = habitos['Tempo_Midias_Sociais_Minutos'].fillna(0)
habitos['Atividade_Extra'] = habitos['Atividade_Extra'].fillna('Não Aplicável')
habitos['Faltas_Anuais'] = habitos['Faltas_Anuais'].fillna(0)

habitos = habitos.astype({
    'ID_Aluno': 'int',
    'Horas_Estudo_Diario': 'float',
    'Tempo_Midias_Sociais_Minutos': 'float',
    'Atividade_Extra': 'str',
    'Faltas_Anuais': 'int'
})

if habitos["ID_Aluno"].duplicated().any():
    duplicados_habitos = habitos[habitos["ID_Aluno"].duplicated(keep=False)]
    print(f"Existem registros duplicados: {duplicados_habitos["ID_Aluno"]}")
else:
    print("Não existem registros duplicados.")

- Base de Dados - Rendimentos

In [ ]:
rendimento = rendimento.astype({
    "ID_Rendimento": "int",
    "ID_Aluno": "int",
    "Disciplina": "str",
    "Nota_Final": "float"
})

if rendimento["ID_Aluno"].duplicated().any() and rendimento["ID_Rendimento"].duplicated().any():
    duplicados_rendimento = rendimento[rendimento["ID_Aluno"].duplicated(keep=False)], rendimento[rendimento["ID_Rendimento"].duplicated(keep=False)]
    print(f"Existem registros duplicados: {duplicados_rendimento["ID_Aluno",["ID_Rendimento"]]}")
else:
    print("Não existem registros duplicados.")

- Base que une rendimento, hábito e alunos

In [ ]:
faltas_reprovacao = pd.merge(rendimento, habitos, on="ID_Aluno", how="left") # join entre a base de rendimento e hábitos
faltas_reprovacao = pd.merge(faltas_reprovacao, alunos, on="ID_Aluno", how="left") # resultado do join anterior e join com a base de alunos


def classifica_nota_final(linha):
    if linha["Nota_Final"] >= 6.0 and linha["Faltas_Anuais"] < 30:
        return "Aprovado"
    else:
        return "Reprovado"


faltas_reprovacao["Status"] = faltas_reprovacao.apply(classifica_nota_final, axis=1) ## cria uma coluna nova com base na função


nomes_status = faltas_reprovacao.groupby("ID_Aluno").agg(
    Nome  = ("Nome", "first"),
    Media_Aluno = ("Nota_Final", "mean"),
    Faltas_Anuais = ("Faltas_Anuais", "first"),
    Status = ("Status", "first"),
    Atividade_Extra = ("Atividade_Extra", "first")
    ).reset_index()
## faz um group by e depois aplica um cálculo em cada uma das colunas já agrupadas, sempre usar o group by para agrupar pela PK

 Base que une habitos, rendimentos e escolas

In [ ]:
alunos_escolas = pd.merge(
    escolas[["ID_Escola", "Tipo_Escola", "Localizacao"]],
    alunos[["ID_Aluno", "ID_Escola", "Data_Nascimento"]],
    on="ID_Escola",
    how="left"
    ) # join entre a base de rendimento e hábitos



hab_rend_alunos = pd.merge(
    nomes_status[["Media_Aluno", "ID_Aluno", "Faltas_Anuais", "Status", "Atividade_Extra"]],
    alunos_escolas[["ID_Aluno", "Tipo_Escola", "Localizacao", "Data_Nascimento"]],
    on="ID_Aluno",
    how="left"
    ) # join entre a base de rendimento e hábitos


- Quantidade de alunos reprovados por média, falta e ambos

In [ ]:
total = len(nomes_status)

total_faltas_reprovado = sum(
       nomes_status.groupby("ID_Aluno")["Faltas_Anuais"].apply(lambda x: (x >= 30).any()))

total_media_reprovado = sum(
    nomes_status.groupby("ID_Aluno")["Media_Aluno"].apply(lambda x: (x < 6.0).any())
)

total_media_falta_reprovado = sum(
    nomes_status.groupby("ID_Aluno")["Media_Aluno"].apply(lambda x: (x < 6.0).any())
    ) & sum(nomes_status.groupby("ID_Aluno")["Faltas_Anuais"].apply(lambda x: (x >= 30).any()))

total_aprovado =  sum(nomes_status["Status"] == "Aprovado")
total_reprovado = sum(nomes_status["Status"] == "Reprovado")

print(f"Total Geral de aprovados: {total_aprovado}, {(total_aprovado/total)*100:.2f}%")
print(f"Total Geral de reprovados: {total_reprovado}, {(total_reprovado/total)*100:.2f}%")
print(f"Total de alunos reprovados por falta: {total_faltas_reprovado}, {(total_faltas_reprovado/total_reprovado)*100:.2f}%")
print(f"Total de alunos reprovados por média: {total_media_reprovado}, {(total_media_reprovado/total_reprovado)*100:.2f}%")
print(f"Total de alunos reprovados por falta e média: {total_media_falta_reprovado}, {(total_media_falta_reprovado/total_reprovado)*100:.2f}%")

- Habitos e rendimentos dos alunos de escolas públicas e privadas
- Habitos e rendimentos dos alunos de zona rural e urbana

In [ ]:
escola_reprovado = hab_rend_alunos.groupby("Tipo_Escola").agg(
    Aprovados = ("Status", lambda x: (x == "Aprovado").sum()) ,
    Reprovados = ("Status", lambda x: (x == "Reprovado").sum())
    )

habitos_tp_escolas = hab_rend_alunos.groupby("Atividade_Extra").agg(
    Atividade_Extra = ("Atividade_Extra", "first"),
    Total_Alunos = ("ID_Aluno", "count")
).sort_values(by=["Total_Alunos"], ascending= False)

atv_mais_praticada = hab_rend_alunos["Atividade_Extra"].mode().iloc[0]

atv_mais_praticada_escola = hab_rend_alunos.groupby("Tipo_Escola")["Atividade_Extra"].agg(
    lambda x: x.mode().iloc[0]
)

rend_localizacao =  hab_rend_alunos.groupby("Localizacao").agg(
    Aprovados = ("Status", lambda x: (x == "Aprovado").sum()) ,
    Reprovados = ("Status", lambda x: (x == "Reprovado").sum())
    )

print(f"{rend_localizacao}")

- Diferença entre os hábitos e rendimentos dos alunos da geração alpha e da geração z

In [ ]:
hab_rend_alunos["Data_Nascimento"] = pd.to_datetime(
    hab_rend_alunos["Data_Nascimento"],
    dayfirst=True,
    errors="coerce"
)

gen_millenium = hab_rend_alunos["Data_Nascimento"].between("01/01/1981","31/12/1996")
gen_z = hab_rend_alunos["Data_Nascimento"].between("01/01/1997","31/12/2010")
gen_alpha = hab_rend_alunos["Data_Nascimento"].between("01/01/2011","31/12/2026")

hab_rend_alunos.loc[gen_millenium, "Geracao"] = "Millenium"
hab_rend_alunos.loc[gen_alpha, "Geracao"] = "Alpha"
hab_rend_alunos.loc[gen_z, "Geracao"] = "Z"


rend_geracao = hab_rend_alunos.groupby(["Geracao"]).agg(
    Aprovados = ("Status", lambda x: (x == "Aprovado").sum()) ,
    Reprovados = ("Status", lambda x: (x == "Reprovado").sum()),
   )

rend_geracao["Proporcao_Reprovados"] = (
    ((rend_geracao["Reprovados"] / (rend_geracao["Aprovados"] + rend_geracao["Reprovados"])) * 100)
    .round(2)  # opcional, para limitar casas decimais
    .astype(str) + "%"
)

geracao_mais_reprovada = rend_geracao["Proporcao_Reprovados"].idxmax()

maior_proporcao = rend_geracao["Proporcao_Reprovados"].max()

print(f"A geração com maior proporção de reprovados é {geracao_mais_reprovada}, com {maior_proporcao:.2%} de reprovação.")

print(rend_geracao)